In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
!pip install gensim wandb wikipedia-api langchain langchain_text_splitters langchain-community langchain-huggingface faiss-cpu transformers accelerate bitsandbytes --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 923.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 60.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depende

In [1]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import wandb
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Using device: cuda


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
 
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum())
print("\nAnswer label distribution:\n", train_df["answer"].value_counts())
 
train_df["prompt_len"] = train_df["prompt"].astype(str).apply(lambda x: len(x.split()))
print("\nPrompt word-length stats:\n", train_df["prompt_len"].describe())

print(train_df.head(3))

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
TEXT_COLS = ["prompt", "A", "B", "C", "D", "E"]
 
for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

In [ ]:
def tokenize(text):
    return text.split()

In [ ]:
all_sentences = []
for df in [train_df, test_df]:
    for col in TEXT_COLS:
        all_sentences.extend(df[col].apply(tokenize).tolist())
 
EMBED_DIM = 100
 
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=SEED,
)
 
print("Vocabulary size:", len(w2v_model.wv))
w2v_model.save(os.path.join(OUTPUT_DIR, "word2vec.model"))

In [ ]:
def text_to_vector(text, model, dim=EMBED_DIM):
    words = tokenize(text)
    vecs = []

    for word in words:
        if word in model.wv:
            vecs.append(model.wv[word])

    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)

    avg_vector = np.mean(vecs, axis=0)
    return avg_vector.astype(np.float32)

In [ ]:
LABELS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

class MCQDataset(Dataset):

    def __init__(self, df, w2v_model, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.model = w2v_model
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        question_vector = text_to_vector(row["prompt"], self.model)

        features = []

        for option in LABELS:
            option_vector = text_to_vector(row[option], self.model)

            difference = np.abs(question_vector - option_vector)

            feature = np.concatenate((question_vector, option_vector, difference))

            features.append(feature)

        features = np.array(features)

        data = {}
        data["features"] = torch.tensor(features, dtype=torch.float32)

        if self.has_labels:
            answer = LABEL2IDX[row["answer"]]
            data["label"] = torch.tensor(answer, dtype=torch.long)
        else:
            data["id"] = row["id"]

        return data

In [ ]:
class MCQScorer(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
 
    def forward(self, x):
        batch, n_options, dim = x.shape
        x = x.view(batch * n_options, dim)
        scores = self.net(x)
        scores = scores.view(batch, n_options)
        return scores

In [ ]:
def map_at_3(probs, labels):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = []

    for pred, true in zip(top3, labels):
        if true in pred:
            score.append(1 / (np.where(pred == true)[0][0] + 1))
        else:
            score.append(0)

    return np.mean(score)


def run_epoch(model, loader, optimizer, criterion, train=True):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    for batch in loader:

        x = batch["features"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(train):

            output = model(x)
            loss = criterion(output, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)

        all_probs.append(torch.softmax(output, dim=1).cpu().detach().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    loss = total_loss / len(loader.dataset)
    acc = (all_probs.argmax(1) == all_labels).mean()
    map3 = map_at_3(all_probs, all_labels)

    return loss, acc, map3


try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except:
    pass

wandb.login()

wandb.init(
    project="dlgenai-project-26t2",
    config={
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "hidden_dim": 128
    }
)

cfg = wandb.config

train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["answer"]
)

train_loader = DataLoader(
    MCQDataset(train_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    MCQDataset(val_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=False
)

model = MCQScorer(3 * EMBED_DIM, cfg.hidden_dim).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

best_map = 0

for epoch in range(cfg.epochs):

    train_loss, train_acc, train_map = run_epoch(
        model, train_loader, optimizer, criterion, True
    )

    val_loss, val_acc, val_map = run_epoch(
        model, val_loader, optimizer, criterion, False
    )

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_map3": train_map,
        "val_map3": val_map
    })

    print(f"Epoch {epoch+1}  Validation MAP@3 = {val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), OUTPUT_DIR + "/best_model.pt")

wandb.finish()

In [ ]:
model = MCQScorer(3 * EMBED_DIM, 128).to(DEVICE)
model.load_state_dict(torch.load(OUTPUT_DIR + "/best_model.pt", map_location=DEVICE))
model.eval()

test_loader = DataLoader(
    MCQDataset(test_df, w2v_model, has_labels=False),
    batch_size=32,
    shuffle=False
)

ids = []
predictions = []

with torch.no_grad():

    for batch in test_loader:

        x = batch["features"].to(DEVICE)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()
        top3 = np.argsort(-probs, axis=1)[:, :3]

        for i in range(len(top3)):
            ids.append(int(batch["id"][i]))
            predictions.append(" ".join(LABELS[j] for j in top3[i]))

test_df["Prediction_Model_NN"] = predictions
print("NN Model predictions saved...")

In [9]:
import os
import wikipediaapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

wiki = wikipediaapi.Wikipedia(
    user_agent="MyRAGProject/1.0 (singhshikhar8957@gmail.com)",
    language="en"
)

print("Scraping Wikipedia to build the Knowledge Base...")

# wiki_topics = [
#     "Supersymmetric quantum mechanics", "Heisenberg uncertainty principle", "Virtual particle", 
#     "CEERS-93316", "James Webb Space Telescope", "Redshift", "Interstellar medium", 
#     "Carnot heat engine", "Maxwell's demon", "Throttling process", "Kelvin-Helmholtz instability",
#     "Landau-Lifshitz-Gilbert equation", "Magnetic susceptibility", "Memristor", 
#     "Regular polytope", "Erlangen program", "Lorentz covariance", 
#     "Giordano Bruno", "Radiometric dating", "Cyclida", "Quantum field theory", 
#     "Supermassive black hole", "Main sequence", "Pulsar", "Bernoulli's principle", "Kutta condition", 
#     "Resistivity", "Superconductivity", "Memristor", "Hyperbolic geometry", "Symmetry group", 
#     "Triskeles", "API gravity"
# ]

wiki_topics = [
    "Supersymmetric quantum mechanics", "Heisenberg uncertainty principle", "Virtual particle", 
    "Spontaneous symmetry breaking", "Lorentz covariance", "Wigner distribution function", 
    "Magnetic monopole", "Spin quantum number", "Parity (physics)", "Peierls bracket", 
    "Geometric quantization", "Quantum field theory", "Classical mechanics", "Minkowski space", 
    "Minkowski diagram", "De Haas–van Alphen effect", "Josephson effect", "Earnshaw's theorem", 
    "CEERS-93316", "James Webb Space Telescope", "Metric expansion of space", "Redshift", 
    "Proper distance", "Interstellar medium", "Molecular cloud", "Supernova remnant", 
    "Main sequence", "Pulsar", "Supermassive black hole", "Dark matter", "Gravitational wave", 
    "Doppler effect", "Roche limit", "Gravity Probe B", "Gravitomagnetism", "Inflaton", 
    "Baryon acoustic oscillations", "Modified Newtonian dynamics", "Lyman-alpha line", 
    "Black hole information paradox", "Planetary system", "X-ray pulsar-based navigation", 
    "Rayleigh scattering", "Maxwell's demon", "Carnot heat engine", "Throttling process", 
    "Kelvin-Helmholtz instability", "Coherent turbulent structure", "Cavitation", "Convection", 
    "Fermat's principle", "Emissivity", "Illuminance", "Dielectric loss", "Ultraviolet catastrophe",
    "Leidenfrost effect", "Coffee ring effect", "Droste effect", "Water hammer", 
    "Optical signal-to-noise ratio", "Young's interference experiment", "Bernoulli's principle", "Kutta condition", 
    "Landau-Lifshitz-Gilbert equation", "Magnetic susceptibility", "Memristor", "Pelorism",
    "Spin valve", "Electrical resistivity and conductivity", "Superconductivity", 
    "Amorphous metal", "Variable-range hopping", "Piezoelectricity", "API gravity", 
    "Crystallinity", "Pycnometer", "Evans balance", "Chemical potential", "Radiometric dating", 
    "Fischer–Tropsch process", "Three moment theorem", "Bollard pull", "Ring-imaging Cherenkov detector", 
    "Formal system", "Uniform tilings in hyperbolic plane", "Regular polytope", 
    "Probability amplitude", "Parity of a permutation", "Probability density function", "Reciprocal length", 
    "Symmetry group", "Erlangen program", "Hyperbolic geometry", "Crystallographic point group", 
    "Permutation groups", "Inversion (discrete mathematics)", "Surgical pathology", "Active transport", "Cyclida", 
    "Phageome", "Trophic level", "Pulmonary circulation", "Mammary gland", "Macromolecule", 
    "Organography", "Second", "Coordinated Universal Time", "Triskelion", "Newton's laws of motion", 
    "Right-hand rule", "Giordano Bruno", "Shower-curtain effect", "Wilson cloud", 
    "Ozma problem", "Horror vacui", "Butterfly effect", "Gauss's law", "Scale (map)",
    "Martin Heidegger"
]

scraped_texts = []

for topic in wiki_topics:
    try:
        page = wiki.page(topic)
        if page.exists():
            scraped_texts.append(page.text)
        else:
            print(f"Skipped {topic}: page not found")
    except Exception as e:
        print(f"Skipped {topic} due to error: {e}")
print(f"Successfully scraped {len(scraped_texts)} articles.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.create_documents(scraped_texts)
print(f"Created {len(docs)} chunks.")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", 
                                   model_kwargs={"device": DEVICE}
)
vector_db = FAISS.from_documents(docs, embeddings)

FAISS_SAVE_PATH = "/kaggle/working/faiss_index"
vector_db.save_local(FAISS_SAVE_PATH)
print(f"FAISS index created successfully and saved to {FAISS_SAVE_PATH}")

Scraping Wikipedia to build the Knowledge Base...
Successfully scraped 119 articles.
Created 10780 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index created successfully and saved to /kaggle/working/faiss_index
